# 01 — Ingest NYC TLC data into Bronze

**Before running:** attach `lh_nyc_taxi_contracts` as this notebook's default Lakehouse. This notebook downloads one monthly official TLC Parquet file and the official zone lookup CSV, stores them in `Files/landing/`, and loads Bronze Delta tables. It overwrites the demo tables on rerun.


In [ ]:
from pathlib import Path
import requests
from pyspark.sql import functions as F

BASE = Path("/lakehouse/default/Files")
LANDING = BASE / "landing"
LANDING.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
}

for filename, url in SOURCES.items():
    destination = LANDING / filename
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already downloaded: {destination.name} ({destination.stat().st_size:,} bytes)")
        continue
    temp_destination = destination.with_suffix(destination.suffix + ".part")
    print(f"Downloading {filename} ...")
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with open(temp_destination, "wb") as target:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    target.write(chunk)
    temp_destination.replace(destination)
    print(f"Saved {destination} ({destination.stat().st_size:,} bytes)")

trips_path = str(LANDING / "yellow_tripdata_2024-01.parquet")
zones_path = str(LANDING / "taxi_zone_lookup.csv")

raw_trips = spark.read.parquet(trips_path)
raw_trips = (raw_trips
    .withColumn("_source_file", F.lit("yellow_tripdata_2024-01.parquet"))
    .withColumn("_ingested_at", F.current_timestamp()))

(raw_trips.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_yellow_taxi_trips"))

zone_lookup = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zones_path))
(zone_lookup.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_taxi_zone_lookup"))

print("Bronze trip rows:", spark.table("bronze_yellow_taxi_trips").count())
print("Taxi-zone lookup rows:", spark.table("bronze_taxi_zone_lookup").count())
print("Trip source columns:", spark.table("bronze_yellow_taxi_trips").columns)
display(spark.table("bronze_yellow_taxi_trips").limit(10))
display(spark.table("bronze_taxi_zone_lookup").limit(10))
